# 03 - Visualización Geoespacial e Impacto por Zona

In [ ]:
import pandas as pd
import folium
from folium.plugins import HeatMap
import numpy as np
from math import radians, cos, sin, asin, sqrt

def haversine(lon1, lat1, lon2, lat2):
    lon1, lat1, lon2, lat2 = map(radians, [lon1, lat1, lon2, lat2])
    dlon = lon2 - lon1 
    dlat = lat2 - lat1 
    a = sin(dlat/2)**2 + cos(lat1) * cos(lat2) * sin(dlon/2)**2
    c = 2 * asin(sqrt(a)) 
    return c * 6371 # km

# Clases en el mismo orden que el modelo (class_id 0–8)
CLASSES = ["Horn", "Siren", "Pets", "Physiological", "Speech", "Ring Tone", "Vibrating", "Notifications", "Cry"]
CLASS_COLORS = {"Horn": "red", "Siren": "darkred", "Speech": "blue", "Ring Tone": "cadetblue",
                "Cry": "purple", "Pets": "green", "Physiological": "orange", "Vibrating": "gray", "Notifications": "black"}
center = [39.47, -0.37]

# Carga de datos
pred_geo = pd.read_parquet("../data/processed/predictions_filtered.parquet")
pred_raw = pd.read_parquet("../data/processed/predictions_geo.parquet")
tracks = pd.read_parquet("../data/processed/tracks.parquet")

if 'class_name' not in pred_geo.columns:
    pred_geo['class_name'] = pred_geo['class'].apply(lambda x: CLASSES[int(x)])
if 'class_name' not in pred_raw.columns:
    pred_raw['class_name'] = pred_raw['class'].apply(lambda x: CLASSES[int(x)])

# --- Pre-procesamiento de trayectorias (distancia acumulada) ---
print("Procesando trayectorias multitemporales...")
groups = []
for date_val, grp in tracks.groupby('date'):
    grp = grp.sort_values('time').copy()
    coords = grp[['lat', 'lon']].values
    dists = [0.0]
    for k in range(1, len(coords)):
        dists.append(haversine(coords[k-1][1], coords[k-1][0], coords[k][1], coords[k][0]))
    grp['cum_dist'] = np.cumsum(dists)
    groups.append(grp)
tracks = pd.concat(groups, ignore_index=True)

# Trayectoria de referencia (primer día) para compatibilidad con celdas antiguas
first_date = sorted(tracks['date'].unique())[0]
ref_track = tracks[tracks['date'] == first_date].copy()
max_s = int(ref_track['cum_dist'].max())

print(f"Eventos filtrados: {len(pred_geo)} | Totales (raw): {len(pred_raw)}")
print(f"Total días procesados: {tracks['date'].nunique()}")
print(f"Trayectoria de referencia: {ref_track['cum_dist'].max():.2f} km")

## 1. DATOS FILTRADOS (CONFIANZA ALTA)

### 1.1 Mapa de Calor

In [1]:
m_heat_f = folium.Map(location=center, zoom_start=13, tiles='CartoDB positron')
HeatMap(pred_geo[['lat', 'lon', 'confidence']].values.tolist(), radius=15, blur=10, min_opacity=0.4).add_to(m_heat_f)
m_heat_f

NameError: name 'folium' is not defined

### 1.2 Capas por Clase

In [ ]:
m_layers_f = folium.Map(location=center, zoom_start=13, tiles='CartoDB positron')
for cls in pred_geo['class_name'].unique():
    group = folium.FeatureGroup(name=cls)
    subset = pred_geo[pred_geo['class_name'] == cls]
    for _, row in subset.iterrows():
        folium.CircleMarker(location=[row['lat'], row['lon']], radius=5, color=CLASS_COLORS.get(cls, 'gray'),
                            fill=True, fill_opacity=0.6, popup=f'{cls} ({row["confidence"]:.2f})').add_to(group)
    group.add_to(m_layers_f)
folium.LayerControl().add_to(m_layers_f)
m_layers_f

### 1.3 Densidad por KM (Semáforo)

In [22]:
# 1.3 Densidad por Tramos de 1km (Filtrado)
events = pred_geo.sort_values('t_start').copy()
events['date'] = events['t_start'].dt.tz_convert('UTC').dt.strftime('%d-%m-%Y')
tracks_sorted = tracks.sort_values('time').copy()
events = pd.merge_asof(events, tracks_sorted[['time', 'cum_dist', 'date']], left_on='t_start', right_on='time', by='date', direction='nearest')

events['segment'] = (events['cum_dist'] // 1.0).astype(int)
counts = events.groupby('segment').size()

# Creamos DataFrame resumen
all_segments = pd.Index(range(max_s + 1), name='Segmento (KM)')
df_counts_f = pd.DataFrame({'Eventos': counts}).reindex(all_segments, fill_value=0)

q1, q2, q3 = df_counts_f['Eventos'].quantile(0.25), df_counts_f['Eventos'].quantile(0.50), df_counts_f['Eventos'].quantile(0.75)

colors = []
for s in df_counts_f.index:
    c = df_counts_f.at[s, 'Eventos']
    colors.append('green' if c <= q1 else 'yellow' if c <= q2 else 'orange' if c <= q3 else 'red')
df_counts_f['Color'] = colors

# Añadimos fila de Total
df_counts_f.loc['Total'] = df_counts_f.sum(numeric_only=True)
df_counts_f.at['Total', 'Color'] = '-'

from IPython.display import display
print("=== Tabla de Conteos por Tramo (Filtrado) ===")
display(df_counts_f)

m_seg_f = folium.Map(location=center, zoom_start=13, tiles='CartoDB positron')
for s in range(max_s + 1):
    c = df_counts_f.at[s, 'Eventos']
    if s == 'Total': continue
    color = df_counts_f.at[s, 'Color']
    seg_pts = ref_track[(ref_track['cum_dist'] >= s) & (ref_track['cum_dist'] < s+1)]
    if not seg_pts.empty:
        folium.PolyLine(seg_pts[['lat', 'lon']].values.tolist(), color=color, weight=6, opacity=0.8, 
                        popup=f'KM {s}-{s+1}: {c} eventos ({color})').add_to(m_seg_f)

m_seg_f


## 2. DATOS RAW (TODAS LAS DETECCIONES)

### 2.1 Mapa de Calor (Raw)

In [ ]:
m_heat_r = folium.Map(location=center, zoom_start=13, tiles='CartoDB dark_matter')
HeatMap(pred_raw[['lat', 'lon', 'confidence']].values.tolist(), radius=12, blur=8, min_opacity=0.3).add_to(m_heat_r)
m_heat_r

### 2.2 Capas por Clase (Raw)

In [ ]:
m_layers_r = folium.Map(location=center, zoom_start=13, tiles='CartoDB positron')
for cls in pred_raw['class_name'].unique():
    group = folium.FeatureGroup(name=cls)
    subset = pred_raw[pred_raw['class_name'] == cls]
    if len(subset) > 500: subset = subset.sample(500, random_state=42)
    for _, row in subset.iterrows():
        folium.CircleMarker(location=[row['lat'], row['lon']], radius=3, color=CLASS_COLORS.get(cls, 'gray'),
                            fill=True, fill_opacity=0.4, popup=f'{cls} ({row["confidence"]:.2f})').add_to(group)
    group.add_to(m_layers_r)
folium.LayerControl().add_to(m_layers_r)
m_layers_r

### 2.3 Densidad por KM (Semáforo Raw)

In [29]:
# 2.3 Densidad por Tramos de 1km (Raw)
events_r = pred_raw.sort_values('t_start').copy()
events_r['date'] = events_r['t_start'].dt.tz_convert('UTC').dt.strftime('%d-%m-%Y')
tracks_sorted = tracks.sort_values('time').copy()
events_r = pd.merge_asof(events_r, tracks_sorted[['time', 'cum_dist', 'date']], left_on='t_start', right_on='time', by='date', direction='nearest')

events_r['segment'] = (events_r['cum_dist'] // 1.0).astype(int)
counts_r = events_r.groupby('segment').size()

all_segments = pd.Index(range(max_s + 1), name='Segmento (KM)')
df_counts_r = pd.DataFrame({'Eventos': counts_r}).reindex(all_segments, fill_value=0)

q1_r, q2_r, q3_r = df_counts_r['Eventos'].quantile(0.25), df_counts_r['Eventos'].quantile(0.50), df_counts_r['Eventos'].quantile(0.75)

colors_r = []
for s in df_counts_r.index:
    c = df_counts_r.at[s, 'Eventos']
    colors_r.append('green' if c <= q1_r else 'yellow' if c <= q2_r else 'orange' if c <= q3_r else 'red')
df_counts_r['Color'] = colors_r

df_counts_r.loc['Total'] = df_counts_r.sum(numeric_only=True)
df_counts_r.at['Total', 'Color'] = '-'

from IPython.display import display
print("=== Tabla de Conteos por Tramo (Raw) ===")
display(df_counts_r)

m_seg_r = folium.Map(location=center, zoom_start=13, tiles='CartoDB dark_matter')
for s in range(max_s + 1):
    c = df_counts_r.at[s, 'Eventos']
    if s == 'Total': continue
    color = df_counts_r.at[s, 'Color']
    seg_pts = ref_track[(ref_track['cum_dist'] >= s) & (ref_track['cum_dist'] < s+1)]
    if not seg_pts.empty:
        folium.PolyLine(seg_pts[['lat', 'lon']].values.tolist(), color=color, weight=6, opacity=0.8, 
                        popup=f'KM {s}-{s+1}: {c} eventos raw ({color})').add_to(m_seg_r)

m_seg_r


=== Tabla de Conteos por Tramo (Raw) ===


,Eventos,Color
Segmento (KM),,
0,630.0,red
1,531.0,red
2,398.0,orange
3,325.0,yellow
4,202.0,yellow
5,315.0,yellow
6,391.0,orange
7,695.0,red
8,339.0,orange


### 2.4 Análisis de Sensibilidad por Confianza

En esta sección iteramos sobre diferentes umbrales de confianza (desde 0.1 hasta 0.9) para analizar cómo varía la cantidad de detecciones por segmento. Calculamos la media de estos conteos para cada tramo. Dado que el primer (KM 0) y último kilómetro suelen tener anomalías (exceso de ruido por inicio/fin de trayecto o coche parado), se colorean de gris y se excluyen del cálculo estadístico de los cuartiles del semáforo.

In [30]:
import numpy as np
from IPython.display import display

# Cruce de datos multitemporal correcto por fecha
events_all = pred_raw.sort_values('t_start').copy()
events_all['date'] = events_all['t_start'].dt.tz_convert('UTC').dt.strftime('%d-%m-%Y')
tracks_sorted = tracks.sort_values('time').copy()
events_all = pd.merge_asof(events_all, tracks_sorted[['time', 'cum_dist', 'date']], left_on='t_start', right_on='time', by='date', direction='nearest')
events_all['segment'] = (events_all['cum_dist'] // 1.0).astype(int)

# Iteramos sobre los umbrales de confianza
thresholds = np.arange(0.1, 1.0, 0.1)
counts_dict = {}

for th in thresholds:
    filtered = events_all[events_all['confidence'] >= th]
    counts_dict[f"{th:.1f}"] = filtered.groupby('segment').size()

# Creamos el DataFrame resumen
df_counts = pd.DataFrame(counts_dict).fillna(0).astype(int)

# Aseguramos que todos los segmentos posibles (0 a max_s) estén en el índice
all_segments = pd.Index(range(max_s + 1), name='Segmento (KM)')
df_counts = df_counts.reindex(all_segments, fill_value=0)

# Calculamos la media de cada tramo (por fila)
df_counts['Media'] = df_counts.mean(axis=1).round(1)

# Calculamos los colores basados en la Media usando Cuartiles
q1_mean, q2_mean, q3_mean = df_counts['Media'].quantile(0.25), df_counts['Media'].quantile(0.50), df_counts['Media'].quantile(0.75)

colors = []
for s in df_counts.index:
    c = df_counts.at[s, 'Media']
    colors.append('green' if c <= q1_mean else 'yellow' if c <= q2_mean else 'orange' if c <= q3_mean else 'red')
df_counts['Color'] = colors

# Añadimos fila de Total
df_counts.loc['Total'] = df_counts.sum(numeric_only=True)
df_counts.at['Total', 'Color'] = '-'

# Imprimimos la tabla
print("=== Tabla de Conteos por Tramo y Umbral de Confianza ===")
display(df_counts)

# Mapa tipo 2.3 (Semáforo)
m_seg_mean = folium.Map(location=center, zoom_start=13, tiles='CartoDB dark_matter')
for s in range(max_s + 1):
    if s == 'Total': continue
    c = df_counts.at[s, 'Media']
    color = df_counts.at[s, 'Color']
    popup_text = f'KM {s}-{s+1}: Media {c} ({color})'
        
    seg_pts = ref_track[(ref_track['cum_dist'] >= s) & (ref_track['cum_dist'] < s+1)]
    if not seg_pts.empty:
        folium.PolyLine(seg_pts[['lat', 'lon']].values.tolist(), color=color, weight=6, opacity=0.8, 
                        popup=popup_text).add_to(m_seg_mean)

m_seg_mean


=== Tabla de Conteos por Tramo y Umbral de Confianza ===


,0.1,0.2,0.3,0.4,0.5,0.6,0.7,0.8,0.9,Media,Color
Segmento (KM),,,,,,,,,,,
0,630.0,395.0,276.0,180.0,119.0,77.0,47.0,22.0,4.0,194.4,red
1,531.0,304.0,204.0,146.0,95.0,61.0,41.0,24.0,5.0,156.8,red
2,398.0,253.0,176.0,125.0,93.0,69.0,33.0,14.0,2.0,129.2,orange
3,325.0,193.0,132.0,101.0,72.0,56.0,38.0,19.0,7.0,104.8,orange
4,202.0,120.0,89.0,61.0,46.0,34.0,20.0,11.0,0.0,64.8,yellow
5,315.0,195.0,136.0,96.0,68.0,45.0,15.0,9.0,1.0,97.8,yellow
6,391.0,245.0,163.0,106.0,77.0,53.0,37.0,21.0,8.0,122.3,orange
7,695.0,454.0,322.0,241.0,183.0,121.0,79.0,43.0,10.0,238.7,red
8,339.0,185.0,124.0,80.0,49.0,31.0,11.0,2.0,1.0,91.3,yellow


## 2.5 Análisis de Velocidad de Desplazamiento

En esta sección calculamos la velocidad de la trayectoria para comparar visualmente los 'hotspots' acústicos con la velocidad de circulación. Esto permite identificar si los distractores ocurren predominantemente en zonas de baja velocidad (posible congestión o zonas urbanas densas) o alta velocidad.

In [36]:
import branca.colormap as cm

# 1. Cálculo de velocidad punto a punto
ref_track['time_dt'] = pd.to_datetime(ref_track['time'])
ref_track['time_diff'] = ref_track['time_dt'].diff().dt.total_seconds()
ref_track['dist_diff'] = ref_track['cum_dist'].diff() * 1000  # Convertir km a metros

# Velocidad (km/h) = (m/s) * 3.6
ref_track['speed_kmh'] = (ref_track['dist_diff'] / ref_track['time_diff'].replace(0, np.nan)) * 3.6

# 2. Limpieza y Suavizado
# Filtramos picos irreales por saltos de GPS (>120 km/h en ciudad)
ref_track['speed_kmh'] = ref_track['speed_kmh'].fillna(0).clip(0, 100)
# Suavizado con media móvil para visualización más limpia
ref_track['speed_smooth'] = ref_track['speed_kmh'].rolling(window=7, min_periods=1, center=True).mean()

print(f"Velocidad máxima registrada (suavizada): {ref_track['speed_smooth'].max():.2f} km/h")
print(f"Velocidad media de la ruta: {ref_track['speed_smooth'].mean():.2f} km/h")

# 3. Visualización en Mapa
m_speed = folium.Map(location=center, zoom_start=13, tiles='CartoDB positron')

# Definimos escala de colores: Verde (Rápido) -> Rojo (Lento/Parado)
colormap = cm.linear.RdYlGn_09.scale(0, 50)  # Escala de 0 a 50 km/h para entorno urbano
colormap.caption = 'Velocidad de desplazamiento (km/h)'

# Dibujamos la trayectoria segmento a segmento
for i in range(1, len(ref_track)):
    p1 = ref_track.iloc[i-1]
    p2 = ref_track.iloc[i]
    speed = p2['speed_smooth']
    
    if pd.isna(speed): continue
    
    folium.PolyLine(
        locations=[[p1['lat'], p1['lon']], [p2['lat'], p2['lon']]],
        color=colormap(speed),
        weight=5,
        opacity=0.9,
        tooltip=f"{speed:.1f} km/h"
    ).add_to(m_speed)

colormap.add_to(m_speed)
m_speed

Velocidad máxima registrada (suavizada): 82.00 km/h
Velocidad media de la ruta: 50.35 km/h


## 3. Mapa de Calor — Sólo Horn + Siren (Eventos Peligrosos)
Riesgo vial concentrado: solo claxon y sirenas. Rojo = alta conflictividad.

In [ ]:
danger_raw = pred_raw[pred_raw['class_name'].isin(['Horn','Siren'])].dropna(subset=['lat','lon'])
danger_fil = pred_geo[pred_geo['class_name'].isin(['Horn','Siren'])].dropna(subset=['lat','lon'])

m_danger = folium.Map(location=center, zoom_start=13, tiles='CartoDB dark_matter')
HeatMap(danger_raw[['lat','lon','confidence']].values.tolist(),
        radius=18, blur=12, min_opacity=0.5,
        gradient={0.4:'blue',0.6:'orange',0.8:'red'}).add_to(m_danger)
print(f'Horn+Siren raw: {len(danger_raw)} | filtrados: {len(danger_fil)}')
m_danger


## 4. Danger Score — Ruta Coloreada por Peligrosidad
Carga danger_scores.parquet de NB02b. Verde=seguro, Rojo=peligroso.

In [ ]:
import branca.colormap as cm
from pathlib import Path as _P
_dp = _P('../data/processed')
if (_dp / 'danger_scores.parquet').exists():
    df_danger = pd.read_parquet(_dp / 'danger_scores.parquet')
    max_score = df_danger['danger_score'].max() or 1
    cmap_d = cm.linear.RdYlGn_09.scale(max_score, 0)  # invertido: alto=rojo
    cmap_d.caption = 'Danger Score (Horn + 2×Siren)'
    m_dscore = folium.Map(location=center, zoom_start=13, tiles='CartoDB positron')
    for s in df_danger.index:
        seg_pts = ref_track[(ref_track['cum_dist'] >= s) & (ref_track['cum_dist'] < s+1)]
        if seg_pts.empty: continue
        score = df_danger.at[s, 'danger_score']
        tip = f'KM {s}-{s+1} | Score: {score:.0f} | Horn: {df_danger.at[s, "horn"]:.0f} | Siren: {df_danger.at[s, "siren"]:.0f}'
        folium.PolyLine(seg_pts[['lat','lon']].values.tolist(),
                        color=cmap_d(score), weight=8, opacity=0.9, tooltip=tip).add_to(m_dscore)
    cmap_d.add_to(m_dscore)
    display(m_dscore)
else:
    print('[WARN] Ejecutar 02b_analysis_gps_danger.ipynb primero para generar danger_scores.parquet')


## 5. Heatmaps por Clase Acústica
Un mapa HTML por clase. Guardados en outputs/maps_by_class/.

In [ ]:
import math, os
out_dir = Path('../outputs/maps_by_class')
out_dir.mkdir(parents=True, exist_ok=True)
classes_ok = [c for c in CLASSES
              if len(pred_raw[pred_raw['class_name']==c].dropna(subset=['lat','lon'])) > 10]
for cls in classes_ok:
    subset = pred_raw[pred_raw['class_name']==cls].dropna(subset=['lat','lon'])
    m_cls = folium.Map(location=center, zoom_start=13, tiles='CartoDB positron')
    HeatMap(subset[['lat','lon','confidence']].values.tolist(),
            radius=15, blur=10, min_opacity=0.4).add_to(m_cls)
    folium.Marker(center, popup=f'{cls} — {len(subset)} eventos',
                  icon=folium.Icon(color='darkred')).add_to(m_cls)
    m_cls.save(str(out_dir / f'heatmap_{cls}.html'))
print(f'Mapas guardados: {classes_ok}')
# Vista previa de la primera clase
if classes_ok:
    subset = pred_raw[pred_raw['class_name']==classes_ok[0]].dropna(subset=['lat','lon'])
    m_prev = folium.Map(location=center, zoom_start=13, tiles='CartoDB positron')
    HeatMap(subset[['lat','lon','confidence']].values.tolist(), radius=15).add_to(m_prev)
    display(m_prev)


## 6. Cluster de Marcadores Interactivo
Click en cada cluster para ver clase, confianza, fecha y hora.

In [ ]:
from folium.plugins import MarkerCluster
m_cluster = folium.Map(location=center, zoom_start=13, tiles='CartoDB positron')
mc = MarkerCluster()
for _, row in pred_geo.dropna(subset=['lat','lon']).iterrows():
    color = CLASS_COLORS.get(row['class_name'], 'gray')
    hora = row['t_start'].tz_convert('Europe/Madrid').strftime('%H:%M')
    popup_html = (
        f'<b>{row["class_name"]}</b><br>'
        f'Confianza: {row["confidence"]:.3f}<br>'
        f'Fecha: {row["date"]}<br>'
        f'Hora: {hora} | Mic: {int(row["microfono_id"])}'
    )
    folium.CircleMarker(
        location=[row['lat'], row['lon']], radius=5,
        color=color, fill=True, fill_opacity=0.8,
        popup=folium.Popup(popup_html, max_width=220)
    ).add_to(mc)
mc.add_to(m_cluster)
folium.LayerControl().add_to(m_cluster)
m_cluster


## 7. Contornos KDE — Núcleos de Riesgo
Kernel Density Estimation sobre Horn+Siren para visualizar núcleos como puntos calientes.

In [ ]:
from scipy.stats import gaussian_kde
pts = danger_raw[['lat','lon']].values
if len(pts) >= 10:
    kde = gaussian_kde(pts.T, bw_method=0.08)
    lat_r = np.linspace(pts[:,0].min()-0.01, pts[:,0].max()+0.01, 80)
    lon_r = np.linspace(pts[:,1].min()-0.01, pts[:,1].max()+0.01, 80)
    xx, yy = np.meshgrid(lat_r, lon_r)
    zz = kde(np.vstack([xx.ravel(), yy.ravel()])).reshape(xx.shape)
    threshold = zz.mean() + 1.5 * zz.std()
    hot_idx = np.argwhere(zz >= threshold)
    m_kde = folium.Map(location=center, zoom_start=13, tiles='CartoDB dark_matter')
    HeatMap(pts.tolist(), radius=20, blur=15, min_opacity=0.5).add_to(m_kde)
    for idx in hot_idx[::3]:
        folium.CircleMarker(
            location=[lat_r[idx[1]], lon_r[idx[0]]],
            radius=4, color='white', fill=True, fill_opacity=0.3
        ).add_to(m_kde)
    print(f'KDE: {len(hot_idx)} puntos sobre umbral (media+1.5σ)')
    display(m_kde)
else:
    print(f'[WARN] Pocos puntos para KDE: {len(pts)}')


## 8. Trayectos Móvil vs Sistema Micófonos
Comparativa visual de rutas. Requiere ejecutar scripts/prepare_mobile.py antes.

In [ ]:
_mob_path = Path('../data/processed/tracks_mobile.parquet')
_mob_pred_path = Path('../data/processed/predictions_mobile.parquet')
if _mob_path.exists():
    tracks_mob = pd.read_parquet(_mob_path)
    m_mobile = folium.Map(location=center, zoom_start=13, tiles='CartoDB positron')
    folium.PolyLine(ref_track[['lat','lon']].values.tolist(),
                    color='blue', weight=3, opacity=0.5,
                    tooltip='Sistema micófonos (referencia)').add_to(m_mobile)
    colors_mob = ['orange','green','purple','darkred']
    for i, (sid, grp) in enumerate(tracks_mob.groupby('session_id')):
        grp = grp.sort_values('time')
        folium.PolyLine(grp[['lat','lon']].values.tolist(),
                        color=colors_mob[i % 4], weight=4, opacity=0.8,
                        tooltip=f'Móvil: {sid}').add_to(m_mobile)
    if _mob_pred_path.exists():
        pred_mob = pd.read_parquet(_mob_pred_path)
        if 'class_name' not in pred_mob.columns:
            pred_mob['class_name'] = pred_mob['class_id'].apply(lambda x: CLASSES[int(x)])
        danger_mob = pred_mob[pred_mob['class_name'].isin(['Horn','Siren'])].dropna(subset=['lat','lon'])
        HeatMap(danger_mob[['lat','lon']].values.tolist() if len(danger_mob) else [],
                radius=15, blur=10, min_opacity=0.6).add_to(m_mobile)
    folium.LayerControl().add_to(m_mobile)
    print(f'Sesiones móvil: {list(tracks_mob["session_id"].unique())}')
    display(m_mobile)
else:
    print('[INFO] Sin datos móvil. Ejecutar: python scripts/prepare_mobile.py')


## 9. Zoom en Zonas Peligrosas Conocidas
Mapa centrado en cada zona conocida con todos los layers. Editar KNOWN_ZONES_MAP con las coordenadas reales.

In [ ]:
# ⚠️ CONFIGURACIÓN: editar con las coordenadas reales
KNOWN_ZONES_MAP = [
    {'nombre': 'Zona Peligrosa 1', 'lat': 39.47, 'lon': -0.37},
    {'nombre': 'Zona Peligrosa 2', 'lat': 39.48, 'lon': -0.39},
]
for zone in KNOWN_ZONES_MAP:
    m_zone = folium.Map(location=[zone['lat'], zone['lon']],
                        zoom_start=16, tiles='CartoDB positron')
    # Heatmap Horn+Siren
    HeatMap(danger_raw[['lat','lon','confidence']].values.tolist(),
            radius=22, blur=14, min_opacity=0.5,
            gradient={0.4:'blue',0.65:'orange',0.85:'red'}).add_to(m_zone)
    # Todos los eventos filtrados
    for _, row in pred_geo.dropna(subset=['lat','lon']).iterrows():
        color = CLASS_COLORS.get(row['class_name'], 'gray')
        folium.CircleMarker(
            location=[row['lat'], row['lon']], radius=4,
            color=color, fill=True, fill_opacity=0.7,
            tooltip=f'{row["class_name"]} ({row["confidence"]:.2f})'
        ).add_to(m_zone)
    # Marker de la zona
    folium.Marker(
        location=[zone['lat'], zone['lon']],
        popup=zone['nombre'],
        icon=folium.Icon(color='red', icon='exclamation-sign')
    ).add_to(m_zone)
    print(f"\n=== {zone['nombre']} ===")
    display(m_zone)
